# Assignment 8: Working with Databases 

Write a Python program to create and manage a Student Database. Create a database named `student.db` and a table `Student` with the following fields:

$$
\begin{array}{|c|c|}
\hline
\textbf{Field} & \textbf{Data Type} \\
\hline
\text{Roll\_No} & \text{INTEGER PRIMARY KEY} \\
\text{Name} & \text{TEXT} \\
\text{Course} & \text{TEXT} \\
\text{Marks} & \text{REAL} \\
\text{City} & \text{TEXT} \\
\hline
\end{array}
$$

The program should provide a menu-driven interface to perform the following operations:
1. Create Database and Table
    - Create the database and Student table using SQL.
2. Insert Student 
    - Accept student details from the user. 
    - Insert the record using an SQL `INSERT` query. 
3. Display All Students  
    - Retrieve and display all student records using `SELECT`. 
4. Search Student 
    - Accept Roll Number from the user. 
    - Display the student's details using `SELECT` and `WHERE`. 
5. Update Student 
    - Accept Roll Number and new marks. 
    - Update the student's marks using `UPDATE`. 
6. Delete Student 
    - Accept Roll Number. 
    - Delete the corresponding record using `DELETE`. 
7. Display Students Scoring Above 75 
    - Use an SQL query with `WHERE` Marks > 75. 
8. Exit 
    - Close the database connection and terminate the program. 

### 1. Importing SQL connector for python:

In [2]:
!pip install mysql-connector-python

In [3]:
import mysql.connector
from mysql.connector import Error

# Configure MYSQL connection:

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "Sql@2025",
}
DB_NAME = "student_db"

In [4]:
def get_server_connection():
  """Establishes connection to the MySQL server (without selecting a database)."""
  try:
    return mysql.connector.connect(**DB_CONFIG)
  except Error as err:
    print(f"Error connecting to MySQL Server: {err}")
    return None

In [5]:
def get_db_connection():
  """Establishes connection directly to the student_db database."""
  try:
    config = DB_CONFIG.copy()
    config["database"] = DB_NAME
    return mysql.connector.connect(**config)
  except Error as err:
    print(f"Error connecting to database '{DB_NAME}': {err}")
    return None

### 2. Create Database and Table Function (`create_database_and_table()`):

In [6]:
def create_database_and_table():
  """Creates the database and the Student table with specified fields."""
  # Step 1: Create Database
  conn = get_server_connection()
  if not conn:
    return
  cursor = conn.cursor()
  cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
  print(f"Database '{DB_NAME}' verified/created.")
  cursor.close()
  conn.close()

  # Step 2: Create Table
  conn = get_db_connection()
  if not conn:
    return
  cursor = conn.cursor()
  create_table_query = """
    CREATE TABLE IF NOT EXISTS Student (
        Roll_No INT PRIMARY KEY,
        Name VARCHAR(100) NOT NULL,
        Course VARCHAR(50) NOT NULL,
        Marks FLOAT NOT NULL,
        City VARCHAR(50) NOT NULL
    )
    """
  cursor.execute(create_table_query)
  conn.commit()
  print("Table 'Student' verified/created successfully.")
  cursor.close()
  conn.close()

### 3. Insert student function (`insert_student()`):

In [7]:
def insert_student():
  """Accepts student details from the user and inserts them into the table."""
  conn = get_db_connection()
  if not conn:
    return

  try:
    roll_no = int(input("Enter Roll No: "))
    name = input("Enter Name: ").strip()
    course = input("Enter Course: ").strip()
    marks = float(input("Enter Marks: "))
    city = input("Enter City: ").strip()

    cursor = conn.cursor()
    query = """
        INSERT INTO Student (Roll_No, Name, Course, Marks, City)
        VALUES (%s, %s, %s, %s, %s)
        """
    cursor.execute(query, (roll_no, name, course, marks, city))
    conn.commit()
    print("Record inserted successfully!")
  except mysql.connector.IntegrityError:
    print(f"Error: A student with Roll No {roll_no} already exists.")
  except ValueError:
    print("Error: Invalid numeric input for Roll No or Marks.")
  except Error as err:
    print(f"Database error: {err}")
  finally:
    cursor.close()
    conn.close()

### 4. Display all students function (`display_all_students()`):

In [8]:
def display_all_students():
  """Retrieves and displays all student records."""
  conn = get_db_connection()
  if not conn:
    return

  cursor = conn.cursor()
  cursor.execute("SELECT * FROM Student")
  records = cursor.fetchall()

  if not records:
    print("No records found in the database.")
  else:
    print("\n" + "=" * 65)
    print(
        f"{'Roll No':<10} {'Name':<20} {'Course':<15} {'Marks':<10} {'City':<10}"
    )
    print("=" * 65)
    for row in records:
      print(
          f"{row[0]:<10} {row[1]:<20} {row[2]:<15} {row[3]:<10.2f} {row[4]:<10}"
      )
    print("=" * 65)

  cursor.close()
  conn.close()

### 5. Search a student function (`search_student()`):

In [9]:
def search_student():
  """Searches for a student by Roll Number."""
  conn = get_db_connection()
  if not conn:
    return

  try:
    roll_no = int(input("Enter Roll Number to search: "))
    cursor = conn.cursor()
    query = "SELECT * FROM Student WHERE Roll_No = %s"
    cursor.execute(query, (roll_no,))
    record = cursor.fetchone()

    if record:
      print("\n--- Student Found ---")
      print(f"Roll No : {record[0]}")
      print(f"Name    : {record[1]}")
      print(f"Course  : {record[2]}")
      print(f"Marks   : {record[3]}")
      print(f"City    : {record[4]}")
    else:
      print(f"No student found with Roll Number: {roll_no}")
  except ValueError:
    print("Error: Roll Number must be an integer.")
  except Error as err:
    print(f"Database error: {err}")
  finally:
    cursor.close()
    conn.close()

### 6. Update Marks function (`update_marks()`):

In [10]:
def update_marks():
  """Updates marks of a student based on Roll Number."""
  conn = get_db_connection()
  if not conn:
    return

  try:
    roll_no = int(input("Enter Roll Number to update marks: "))
    new_marks = float(input("Enter New Marks: "))

    cursor = conn.cursor()
    query = "UPDATE Student SET Marks = %s WHERE Roll_No = %s"
    cursor.execute(query, (new_marks, roll_no))
    conn.commit()

    if cursor.rowcount > 0:
      print(f"Marks updated successfully for Roll No {roll_no}.")
    else:
      print(f"No student found with Roll Number: {roll_no}.")
  except ValueError:
    print("Error: Invalid numeric input.")
  except Error as err:
    print(f"Database error: {err}")
  finally:
    cursor.close()
    conn.close()

### 7. Delete a student function (`delete_student()`):

In [11]:
def delete_student():
  """Deletes a student record based on Roll Number."""
  conn = get_db_connection()
  if not conn:
    return

  try:
    roll_no = int(input("Enter Roll Number to delete: "))
    cursor = conn.cursor()
    query = "DELETE FROM Student WHERE Roll_No = %s"
    cursor.execute(query, (roll_no,))
    conn.commit()

    if cursor.rowcount > 0:
      print(f"Record for Roll No {roll_no} deleted successfully.")
    else:
      print(f"No record found with Roll Number: {roll_no}.")
  except ValueError:
    print("Error: Roll Number must be an integer.")
  except Error as err:
    print(f"Database error: {err}")
  finally:
    cursor.close()
    conn.close()

### 8. Display all students with marks above 75 (`display_above_75()`):

In [12]:
def display_above_75():
  """Displays all students scoring above 75 marks."""
  conn = get_db_connection()
  if not conn:
    return

  cursor = conn.cursor()
  query = "SELECT * FROM Student WHERE Marks > 75"
  cursor.execute(query)
  records = cursor.fetchall()

  if not records:
    print("No students found with marks above 75.")
  else:
    print("\n" + "=" * 65)
    print(f"{'Roll No':<10} {'Name':<20} {'Course':<15} {'Marks':<10}")
    print("=" * 65)
    for row in records:
      print(f"{row[0]:<10} {row[1]:<20} {row[2]:<15} {row[3]:<10.2f}")
    print("=" * 65)

  cursor.close()
  conn.close()

### 9. Calling all the function together:

In [13]:
def Student_Menu():
  # Ensure the database and table exist before showing the menu
  create_database_and_table()

  while True:
    print("\n====== STUDENT DATABASE ======")
    print("1. Insert Student")
    print("2. Display All Students")
    print("3. Search Student")
    print("4. Update Marks")
    print("5. Delete Student")
    print("6. Display Students with Marks > 75")
    print("7. Exit")
    print("==============================")

    choice = input("Enter your choice (1-7): ").strip()

    if choice == "1":
      insert_student()
    elif choice == "2":
      display_all_students()
    elif choice == "3":
      search_student()
    elif choice == "4":
      update_marks()
    elif choice == "5":
      delete_student()
    elif choice == "6":
      display_above_75()
    elif choice == "7":
      print("Closing database connections and exiting program. Goodbye!")
      break
    else:
      print("Invalid choice! Please select a valid option (1-7).")

In [14]:
if __name__ == "__main__":
  Student_Menu()

Database 'student_db' verified/created.
Table 'Student' verified/created successfully.

====== STUDENT DATABASE ======
1. Insert Student
2. Display All Students
3. Search Student
4. Update Marks
5. Delete Student
6. Display Students with Marks > 75
7. Exit

Roll No    Name                 Course          Marks      City      
266517     Sumedh Ghadge        MSc_DS          100.00     Pune      
266518     Raj Handique         MSc_DS          100.00     Pune      

====== STUDENT DATABASE ======
1. Insert Student
2. Display All Students
3. Search Student
4. Update Marks
5. Delete Student
6. Display Students with Marks > 75
7. Exit
Closing database connections and exiting program. Goodbye!
